# Demo: RAG Chatbot Tư Vấn Tuyển Sinh FPT School

Notebook này hướng dẫn từng bước xây dựng và kiểm thử hệ thống RAG.

**Kiến trúc:**
```
Câu hỏi → Retriever (FAISS + multilingual-e5-small) → Context → Generator (Qwen2.5-1.5B) → Câu trả lời
```

**Trước khi chạy:** Đảm bảo đã chạy `pip install -r requirements.txt` và đang ở thư mục gốc dự án.

In [ ]:
import os, sys
# Đảm bảo import được các module trong dự án
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('Working dir:', os.getcwd())

---
## Bước 1 – Kiểm tra cấu hình

In [ ]:
from config import (
    EMBEDDING_MODEL, LLM_MODEL, CHUNK_SIZE, CHUNK_OVERLAP,
    TOP_K, MAX_NEW_TOKENS, TEMPERATURE, FAISS_INDEX_PATH, DATA_DIR
)

print('=== CẤU HÌNH HỆ THỐNG ===')
print(f'  Embedding Model  : {EMBEDDING_MODEL}')
print(f'  LLM Model        : {LLM_MODEL}')
print(f'  Chunk Size       : {CHUNK_SIZE}')
print(f'  Chunk Overlap    : {CHUNK_OVERLAP}')
print(f'  Top-K Retrieval  : {TOP_K}')
print(f'  Max New Tokens   : {MAX_NEW_TOKENS}')
print(f'  Temperature      : {TEMPERATURE}')
print(f'  Data Dir         : {DATA_DIR}')
print(f'  FAISS Index Path : {FAISS_INDEX_PATH}')

---
## Bước 2 – Ingest dữ liệu

Đọc file `.txt` / `.pdf` → chia chunk → tạo embedding → lưu FAISS index.

In [ ]:
from pathlib import Path

# Kiểm tra dữ liệu đầu vào
data_files = list(Path(DATA_DIR).glob('*.txt')) + list(Path(DATA_DIR).glob('*.pdf'))
print(f'Tìm thấy {len(data_files)} file trong {DATA_DIR}:')
for f in data_files:
    size_kb = f.stat().st_size / 1024
    print(f'  - {f.name}  ({size_kb:.1f} KB)')

In [ ]:
# Chạy ingest (bỏ qua nếu FAISS index đã tồn tại)
faiss_exists = Path(FAISS_INDEX_PATH).exists()
print(f'FAISS index đã tồn tại: {faiss_exists}')

if not faiss_exists:
    from src.ingest import ingest
    ingest()
else:
    print('Bỏ qua ingest – dùng index có sẵn.')
    print('(Để build lại, xóa thư mục vectorstore/ rồi chạy lại cell này)')

---
## Bước 3 – Demo Retriever

Load FAISS index và tìm kiếm document liên quan cho một câu hỏi mẫu.

In [ ]:
from src.retriever import Retriever

retriever = Retriever()

In [ ]:
query = 'Học phí của trường FPT là bao nhiêu?'
results = retriever.retrieve(query, top_k=3)

print(f'Query: "{query}"')
print(f'Tìm được {len(results)} document(s):\n')

for i, (doc, score) in enumerate(results, 1):
    source = os.path.basename(doc.metadata.get('source', 'unknown'))
    print(f'--- Document {i} | Source: {source} | Score: {score:.4f} ---')
    print(doc.page_content[:300])
    print()

In [ ]:
# Xem context đã được format
context = retriever.format_context(results)
print(context[:800], '...')

---
## Bước 4 – Demo Generator

Load LLM Qwen2.5-1.5B và sinh câu trả lời từ một prompt đơn giản.

> ⚠️ Cell này tải model (~3 GB). Lần đầu sẽ mất 5–15 phút tuỳ kết nối mạng.

In [ ]:
from src.generator import Generator

generator = Generator()

In [ ]:
# Test generator độc lập (không có context)
test_prompt = 'Xin chào, bạn có thể giới thiệu ngắn gọn về bản thân không?'
answer = generator.generate(test_prompt)
print('Prompt:', test_prompt)
print('Answer:', answer)

---
## Bước 5 – Demo RAG Pipeline (Retriever + Generator)

Kết hợp hoàn chỉnh: câu hỏi → retrieve context → build prompt → generate.

In [ ]:
from src.pipeline import RAGPipeline

# Tái sử dụng retriever và generator đã load ở trên
pipeline = RAGPipeline.__new__(RAGPipeline)
pipeline.retriever = retriever
pipeline.generator = generator
print('Pipeline sẵn sàng (dùng lại model đã load).')

In [ ]:
def demo_rag(question: str):
    print('=' * 60)
    print(f'CÂU HỎI: {question}')
    print('=' * 60)
    
    result = pipeline.answer(question)
    
    print('\nCÂU TRẢ LỜI:')
    print(result['answer'])
    
    print('\nNGUỒN THAM KHẢO:')
    for i, src in enumerate(result['sources'], 1):
        print(f'  {i}. {src["source"]} (score: {src["score"]:.4f})')
    print()

In [ ]:
demo_rag('Trường FPT tuyển sinh các cấp học nào?')

In [ ]:
demo_rag('Học phí cấp THPT của trường FPT là bao nhiêu?')

In [ ]:
demo_rag('Hồ sơ tuyển sinh vào lớp 6 cần những gì?')

In [ ]:
demo_rag('Trường FPT có học bổng không? Điều kiện ra sao?')

In [ ]:
demo_rag('Chương trình lập trình và AI tại trường FPT như thế nào?')

---
## Bước 6 – Phân tích Retrieval Score

So sánh độ liên quan của các document với các câu hỏi khác nhau.

In [ ]:
import pandas as pd

test_queries = [
    'Học phí của trường FPT là bao nhiêu?',
    'Điều kiện tuyển sinh vào lớp 10 là gì?',
    'Trường FPT có học bổng không?',
    'Chương trình dạy lập trình như thế nào?',
    'Thời gian tuyển sinh là khi nào?',
]

rows = []
for q in test_queries:
    results = retriever.retrieve(q, top_k=3)
    for rank, (doc, score) in enumerate(results, 1):
        source = os.path.basename(doc.metadata.get('source', 'unknown'))
        rows.append({'Câu hỏi': q[:40] + '...', 'Rank': rank, 'Nguồn': source, 'Score (L2)': round(score, 4)})

df = pd.DataFrame(rows)
df

---
## Bước 7 – Chạy đánh giá đầy đủ

In [ ]:
from src.evaluate import run_evaluation

# quick=True để chạy nhanh hơn (3 câu hỏi)
report = run_evaluation(quick=True)

In [ ]:
# Hiển thị báo cáo dạng bảng
gen_data = [
    {
        'Câu hỏi': r['question'][:50] + '...',
        'Độ dài câu TL': r['answer_length'],
        'Thời gian (s)': r['generation_time_s'],
    }
    for r in report['generation']['per_question']
]
pd.DataFrame(gen_data)

---
## Tổng kết

| Thành phần | Kết quả |
|---|---|
| Embedding | `intfloat/multilingual-e5-small` – hỗ trợ tiếng Việt tốt |
| Vector DB | FAISS – tìm kiếm nhanh, không cần server |
| LLM | `Qwen2.5-1.5B-Instruct` – nhỏ gọn, chạy được local |
| Prompt | Template tiếng Việt + system prompt chuyên biệt |
| UI | Gradio – đơn giản, dễ demo |

**Cải tiến tiếp theo:**
- Thêm nhiều tài liệu vào `data/raw/`
- Thử nghiệm với `CHUNK_SIZE` khác nhau
- Tích hợp re-ranking để cải thiện retrieval
- Thêm tính năng lịch sử hội thoại (conversation memory)